# Stage 6e — Design-related recall harm analysis (registered addendum v1.0)

Implements **"Design-Recall Harm Analysis Plan (Option A)", version 1.0, November 12, 2026**
(OSF addendum; DEVIATIONS.md D9 revised and D17) exactly as written. Run it **once**.

**Question.** Among orthopedic devices cleared through 510(k), is citing a predicate that had a
design-related recall associated with a higher hazard that the citing device is itself later recalled?

**What it does**
1. Validation sample: inter-coder κ (before adjudication), PPV of the FDA design label and share of
   FDA non-design events coded design (after adjudication), Wilson 95% CIs.
2. Builds the device table (population, exposure groups at 510(k) receipt, own-recall outcomes).
3. Primary Cox model, stratified by product code, plus the registered secondary models, bounding
   scenario and hand-code sensitivity analysis. CIs from one cluster bootstrap over manufacturer
   families (1,000 replicates, seed 20261110); every model is refit on the same resamples.
4. Kaplan-Meier cumulative incidence of own recall at 5 and 10 years by exposure group.
5. Applies the registered interpretation rules and writes `data/design_harm_summary.json`.

**Inputs (all frozen, no network):** `snapshot/510k_raw.json.gz`, `data/predicate_edges.csv`
(Rule 2 v2), `data/recall_public_dates.csv` (Stage 6c), `data/recall_links_classified.csv` (Stage 5),
`design_arm/data/recall_events.csv` and `event_device_links.csv` (D1), `design_arm/cause_map.py`,
`firm_families.py`, and the returned validation sheets in `design_arm/validation/`.

**Outputs:** `data/design_harm_devices.csv`, `data/design_harm_km.csv`,
`data/design_harm_summary.json`, `design_arm/validation/validation_final_codes.csv`.

**Runtime:** about 20 to 40 minutes, almost all in the bootstrap. Requires `statsmodels`.

In [1]:
import gzip, json, hashlib, os, sys, time
from math import sqrt
import numpy as np
import pandas as pd
from statsmodels.duration.hazard_regression import PHReg

sys.path.insert(0, "."); sys.path.insert(0, "design_arm")
import firm_families as ff
from cause_map import DESIGN, UNCLEAR

SEED = 20261110          # registered
NBOOT = 1000             # registered
SNAP = pd.Timestamp("2026-08-08")   # registered censoring date (frozen snapshot)
MAXGEN = 3
VAL = "design_arm/validation"
print("statsmodels", __import__("statsmodels").__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

statsmodels 0.14.1 | pandas 2.1.4 | numpy 1.26.4


## 1. Validation sample (registered measures)

The three returned sheets are checked against the SHA-256 fingerprints recorded when they were
received, so the measures are computed from the exact files the coders and adjudicator returned.

In [2]:
SHEETS = {  # file -> SHA-256 recorded on receipt
    "Cord_returned_2026-11-17.xlsx":   "6a97eff12bbee98c0e9babc270bbe2b75859c97d365437dbfc2ca7a63a455d41",
    "Gant_returned_2026-11-18.xlsx":   "dcc04f3f0fbda7e74daf4e2a00bd92fb352ef074a6895d385563fa6848b4a0fa",
    "Tanner_returned_2026-11-18.xlsx": "539d9926ceb35e04502accbddcbc0c10280667c73f3be7af9182d614afa6387f",
}
for f, h in SHEETS.items():
    got = hashlib.sha256(open(os.path.join(VAL, f), "rb").read()).hexdigest()
    assert got == h, f"{f} does not match the fingerprint recorded on receipt"
print("CHECKPOINT  returned sheets match their recorded fingerprints")

cm = pd.read_excel(f"{VAL}/Cord_returned_2026-11-17.xlsx", sheet_name="Coding").set_index("item_id")
gd = pd.read_excel(f"{VAL}/Gant_returned_2026-11-18.xlsx", sheet_name="Coding").set_index("item_id").loc[cm.index]
tl = pd.read_excel(f"{VAL}/Tanner_returned_2026-11-18.xlsx", sheet_name="Adjudication").set_index("item_id")
key = pd.read_csv(f"{VAL}/KEY_do_not_share.csv", dtype={"event": str}).set_index("item_id").loc[cm.index]
assert len(cm) == 120 and cm.design_defect.notna().all() and gd.design_defect.notna().all()

def kappa(a, b):
    cats = sorted(set(a) | set(b)); po = float((a == b).mean())
    pe = sum(float((a == c).mean()) * float((b == c).mean()) for c in cats)
    return po, (po - pe) / (1 - pe)

def wilson(x, n, z=1.959964):
    p = x / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return round(p, 4), round(c - h, 4), round(c + h, 4)

po_d, k_d = kappa(cm.design_defect, gd.design_defect)
po_s, k_s = kappa(cm.scope, gd.scope)
dis_d = cm.design_defect != gd.design_defect; dis_s = cm.scope != gd.scope
assert set(cm.index[dis_d | dis_s]) == set(tl.index), "adjudication sheet does not cover exactly the disagreements"
assert tl.final_design.notna().all() and tl.final_scope.notna().all()

V = pd.DataFrame({"cm_design": cm.design_defect, "gd_design": gd.design_defect,
                  "cm_scope": cm.scope, "gd_scope": gd.scope}, index=cm.index)
V["final_design"] = np.where(~dis_d, V.cm_design, tl.final_design.reindex(V.index))
V["final_scope"] = np.where(~dis_s, V.cm_scope, tl.final_scope.reindex(V.index))
V["hand_design"] = V.final_design.eq("yes")          # registered: unclear after adjudication counts as no
V = V.join(key[["event", "fda_cat", "cause"]])
V.to_csv(f"{VAL}/validation_final_codes.csv")

is_des = V.fda_cat.eq("design")
VALIDATION = {
    "n_events": 120,
    "kappa_design": round(k_d, 3), "agreement_design": [int((~dis_d).sum()), 120],
    "kappa_scope": round(k_s, 3), "agreement_scope": [int((~dis_s).sum()), 120],
    "n_adjudicated": int(len(tl)),
    "ppv_fda_design": [int(V.hand_design[is_des].sum()), int(is_des.sum()), *wilson(V.hand_design[is_des].sum(), is_des.sum())],
    "fda_nondesign_coded_design": [int(V.hand_design[~is_des].sum()), int((~is_des).sum()),
                                   *wilson(V.hand_design[~is_des].sum(), (~is_des).sum())],
    "by_fda_category": {c: [int(V.hand_design[V.fda_cat == c].sum()), int((V.fda_cat == c).sum()),
                            *wilson(V.hand_design[V.fda_cat == c].sum(), (V.fda_cat == c).sum())]
                        for c in ["design", "manufacturing_process", "labeling", "packaging_sterility"]},
    "unclear_after_adjudication": V.index[V.final_design == "unclear"].tolist(),
}
print(json.dumps(VALIDATION, indent=1))

CHECKPOINT  returned sheets match their recorded fingerprints
{
 "n_events": 120,
 "kappa_design": 0.734,
 "agreement_design": [
  103,
  120
 ],
 "kappa_scope": 0.484,
 "agreement_scope": [
  91,
  120
 ],
 "n_adjudicated": 40,
 "ppv_fda_design": [
  51,
  60,
  0.85,
  0.7389,
  0.919
 ],
 "fda_nondesign_coded_design": [
  9,
  60,
  0.15,
  0.081,
  0.2611
 ],
 "by_fda_category": {
  "design": [
   51,
   60,
   0.85,
   0.7389,
   0.919
  ],
  "manufacturing_process": [
   9,
   20,
   0.45,
   0.2582,
   0.6579
  ],
  "labeling": [
   0,
   20,
   0.0,
   0.0,
   0.1611
  ],
  "packaging_sterility": [
   0,
   20,
   0.0,
   0.0,
   0.1611
  ]
 },
 "unclear_after_adjudication": [
  "V021"
 ]
}


## 2. Recall events (one row per FDA event, init at or after 1976)

Event categories come from the D1 table, which applies `cause_map.py` (fixed October 15, 2026).

In [3]:
events = pd.read_csv("design_arm/data/recall_events.csv", dtype={"res_event_number": str})
links = pd.read_csv("design_arm/data/event_device_links.csv", dtype=str)
cls = (pd.read_csv("data/recall_links_classified.csv", dtype=str)
         .drop_duplicates("product_res_number").set_index("product_res_number").recall_class.to_dict())
events["init"] = pd.to_datetime(events.event_date_initiated, format="%Y-%m-%d", errors="coerce")
ok = events.init.notna() & (events.init.dt.year >= 1976)
print(f"CHECKPOINT  events {len(events)} | vague (other_unclear) {int((events.fda_category == UNCLEAR).sum())} "
      f"| dropped (init missing or before 1976) {int((~ok).sum())}")
events = events[ok]
cat = events.set_index("res_event_number").fda_category.to_dict()
init = events.set_index("res_event_number").init.to_dict()
links = links[links.res_event_number.isin(init)]
links["ci_ii"] = links.product_res_number.map(cls).isin(["Class I", "Class II"])
EV = (links.groupby(["k_number", "res_event_number"]).ci_ii.any().reset_index()
           .rename(columns={"k_number": "k", "res_event_number": "event"}))
EV["init"] = EV.event.map(init); EV["cat"] = EV.event.map(cat)
ev_by_dev = {k: g for k, g in EV.groupby("k")}
events_of = {k: set(g.event) for k, g in ev_by_dev.items()}
print(f"CHECKPOINT  device-event links {len(EV)} | devices with a recall {EV.k.nunique()} | events {EV.event.nunique()}")

CHECKPOINT  events 1698 | vague (other_unclear) 387 | dropped (init missing or before 1976) 1
CHECKPOINT  device-event links 2574 | devices with a recall 1691 | events 1697


## 3. Devices, predicates, families

In [4]:
raw = json.load(gzip.open("snapshot/510k_raw.json.gz", "rt"))
dev = pd.DataFrame([{k: r.get(k) for k in ["k_number", "date_received", "decision_date", "applicant", "product_code"]}
                    for r in raw]).drop_duplicates("k_number").set_index("k_number")
dev["rec"] = pd.to_datetime(dev.date_received, errors="coerce")
dev["dec"] = pd.to_datetime(dev.decision_date, errors="coerce")
edges = pd.read_csv("data/predicate_edges.csv", dtype=str)
preds = edges.groupby("device_knumber").predicate_knumber.apply(set).to_dict()
pub = pd.read_csv("data/recall_public_dates.csv", dtype=str)
pub = dict(zip(pub.k_number, pd.to_datetime(pub.recall_public_date)))

firm_ev = (EV.drop_duplicates("event")[["event", "init"]]
             .assign(firm=lambda d: d.event.map(events.set_index("res_event_number").recalling_firm)))
by_crude, by_group = {}, {}
for _, r in firm_ev.iterrows():
    by_crude.setdefault(ff._crude(r.firm), []).append(r.init)
    for g, s in (ff.memberships(r.firm) or []):
        by_group.setdefault(g, []).append((r.init, s))

def fam_groups(name, when):
    return {g for g, s in (ff.memberships(name) or []) if s is None or s <= when}

def firm_prior_recall(applicant, when):
    """Registered: the device's corporate family (as of receipt) had any recall event initiated before receipt."""
    if any(i < when for i in by_crude.get(ff._crude(applicant), [])):
        return True
    for g in fam_groups(applicant, when):
        if any(i < when and (s is None or s <= when) for i, s in by_group.get(g, [])):
            return True
    return False

def fam_key(applicant, when):
    g = sorted(fam_groups(applicant, when)); return g[0] if g else ff._crude(applicant)

def ancestors(k):
    out, frontier = {}, {k}
    for gen in range(1, MAXGEN + 1):
        nxt = set()
        for x in frontier:
            for p in preds.get(x, ()):
                if p not in out and p != k:
                    out[p] = gen; nxt.add(p)
        frontier = nxt
    return out

## 4. Device table

Exposure groups are mutually exclusive and assigned at the 510(k) receipt date:
**design** (cites a predicate with a design-related recall initiated before receipt, whose recall was
public before receipt) > **non-design** (cites a predicate whose recall was public before receipt) >
**distant** (an ancestor 2 or 3 generations back had a recall public before receipt) > **reference**.
The design flag is computed under three labelings: FDA (primary), upper bound (vague events counted
as design), and hand-code (adjudicated codes replace the FDA label for the 120 sampled events).

In [5]:
HAND = dict(zip(V.event, V.hand_design))
def is_design(event, labeling):
    if labeling == "hand" and event in HAND:
        return HAND[event]
    c = cat.get(event)
    return c == DESIGN or (labeling == "upper" and c == UNCLEAR)

def design_init(p, labeling):
    g = ev_by_dev.get(p)
    if g is None: return pd.NaT
    d = g[[is_design(e, labeling) for e in g.event]]
    return d.init.min() if len(d) else pd.NaT

rows, n_bad_dates = [], 0
for k, ps in preds.items():
    if k not in dev.index: continue
    r = dev.loc[k]
    if pd.isna(r.rec) or pd.isna(r.dec):
        n_bad_dates += 1; continue
    anc = ancestors(k)
    gen = min([g for a, g in anc.items() if a in pub and pub[a] < r.rec], default=0)
    qual = [p for p in ps if p in pub and pub[p] < r.rec]           # cited predicates recalled & public before receipt
    row = dict(k=k, applicant=r.applicant, code=str(r.product_code), year=r.dec.year, npred=len(ps), gen=gen,
               prior=firm_prior_recall(r.applicant, r.rec), fam=fam_key(r.applicant, r.rec))
    for lab in ["fda", "upper", "hand"]:
        dq = [p for p in qual if pd.notna(di := design_init(p, lab)) and di < r.rec]
        row[f"design_{lab}"] = bool(dq)
        if lab == "fda":
            row["design_other_fam"] = any(not ff.same_family(r.applicant, dev.applicant.get(p, ""), r.rec) for p in dq)
    shared = set().union(*[events_of.get(a, set()) for a in anc]) if anc else set()
    g = ev_by_dev.get(k)
    own = g[(~g.event.isin(shared)) & (g.init >= r.dec)] if g is not None else EV.iloc[:0]
    for name, sub in [("any", own), ("ci_ii", own[own.ci_ii]), ("design", own[own.cat == DESIGN])]:
        t_end = sub.init.min() if len(sub) else SNAP
        row[f"ev_{name}"] = int(len(sub) > 0); row[f"t_{name}"] = (t_end - r.dec).days / 365.25
    rows.append(row)
D = pd.DataFrame(rows)
n_nonpos = int((D.t_any <= 0).sum())
D = D[D.t_any > 0].reset_index(drop=True)     # follow-up must be positive (cleared on or after the snapshot, or recalled on the clearance day)
for lab in ["fda", "upper", "hand"]:
    D[f"grp_{lab}"] = np.select([D[f"design_{lab}"], D.gen == 1, D.gen.isin([2, 3])],
                                ["design", "nondesign", "distant"], "reference")
D.to_csv("data/design_harm_devices.csv", index=False)
print(f"CHECKPOINT  population {len(D)} devices | excluded: missing dates {n_bad_dates}, non-positive follow-up {n_nonpos}")
print(pd.crosstab(D.grp_fda, D.ev_any, margins=True))

CHECKPOINT  population 8928 devices | excluded: missing dates 0, non-positive follow-up 0
ev_any        0    1   All
grp_fda                   
design      378   60   438
distant    2731  183  2914
nondesign  1377  153  1530
reference  3590  456  4046
All        8076  852  8928


## 5. Cox models and the family-cluster bootstrap

Stratified by product code; Efron ties; covariates: clearance year centered at 2010 per decade with a
quadratic term, log number of predicates, manufacturer prior recall. Every model is fitted on the
same 1,000 family resamples (percentile intervals).

In [6]:
COV = ["yc", "yc2", "lnp", "firm_prior"]
def design_matrix(d, groups):
    x = pd.DataFrame({g: m(d) for g, m in groups.items()}).astype(float)
    x["yc"] = (d.year - 2010) / 10; x["yc2"] = x.yc ** 2; x["lnp"] = np.log(d.npred); x["firm_prior"] = d.prior.astype(float)
    return x

def std_groups(lab):
    return {"design": lambda d: d[f"grp_{lab}"] == "design", "nondesign": lambda d: d[f"grp_{lab}"] == "nondesign",
            "distant": lambda d: d[f"grp_{lab}"] == "distant"}

MODELS = {
    "primary":          ("any", std_groups("fda")),
    "upper_bound":      ("any", std_groups("upper")),
    "hand_code":        ("any", std_groups("hand")),
    "family_split":     ("any", {"design_other_family": lambda d: (d.grp_fda == "design") & d.design_other_fam,
                                 "design_same_family_only": lambda d: (d.grp_fda == "design") & ~d.design_other_fam,
                                 "nondesign": lambda d: d.grp_fda == "nondesign",
                                 "distant": lambda d: d.grp_fda == "distant"}),
    "outcome_class_I_II": ("ci_ii", std_groups("fda")),
    "outcome_design_recall": ("design", std_groups("fda")),
}

def fit(d, outcome, groups):
    x = design_matrix(d, groups)
    m = PHReg(d[f"t_{outcome}"].values, x.values, status=d[f"ev_{outcome}"].values,
              strata=d.code.values, ties="efron").fit(disp=False)
    return pd.Series(m.params, index=x.columns)

point = {name: fit(D, o, g) for name, (o, g) in MODELS.items()}
rng = np.random.default_rng(SEED)
fams = np.array(sorted(D.fam.astype(str).unique()))
idx = {f: np.flatnonzero(D.fam.astype(str).values == f) for f in fams}
boot = {name: [] for name in MODELS}; failed = {name: 0 for name in MODELS}
t0 = time.time()
for b in range(NBOOT):
    d = D.iloc[np.concatenate([idx[f] for f in rng.choice(fams, len(fams), replace=True)])].reset_index(drop=True)
    for name, (o, g) in MODELS.items():
        try:
            boot[name].append(fit(d, o, g))
        except Exception:
            failed[name] += 1
    if (b + 1) % 100 == 0:
        print(f"  bootstrap {b + 1}/{NBOOT}  ({(time.time() - t0) / 60:.1f} min)")

RESULTS = {}
for name, (o, g) in MODELS.items():
    B = pd.DataFrame(boot[name])
    x = design_matrix(D, g); RESULTS[name] = {"outcome": o, "bootstrap_failures": failed[name], "terms": {}}
    for term in list(g) + ["firm_prior"]:
        lo, hi = np.exp(np.nanpercentile(B[term], [2.5, 97.5]))
        n = int(x[term].sum()); e = int(D.loc[x[term] == 1, f"ev_{o}"].sum())
        RESULTS[name]["terms"][term] = {"hr": round(float(np.exp(point[name][term])), 3),
                                        "ci": [round(float(lo), 3), round(float(hi), 3)], "n": n, "events": e}
    print(f"\n{name} (outcome: {o}; failed refits {failed[name]})")
    for term, v in RESULTS[name]["terms"].items():
        print(f"  {term:26} n={v['n']:5}  events={v['events']:4}  HR {v['hr']:.2f} (95% CI {v['ci'][0]:.2f} to {v['ci'][1]:.2f})")

  bootstrap 100/1000  (5.4 min)
  bootstrap 200/1000  (11.0 min)
  bootstrap 300/1000  (17.4 min)
  bootstrap 400/1000  (23.7 min)
  bootstrap 500/1000  (29.9 min)
  bootstrap 600/1000  (35.5 min)
  bootstrap 700/1000  (41.3 min)
  bootstrap 800/1000  (47.0 min)
  bootstrap 900/1000  (52.6 min)
  bootstrap 1000/1000  (58.3 min)

primary (outcome: any; failed refits 0)
  design                     n=  438  events=  60  HR 1.63 (95% CI 1.17 to 2.15)
  nondesign                  n= 1530  events= 153  HR 1.09 (95% CI 0.81 to 1.35)
  distant                    n= 2914  events= 183  HR 0.94 (95% CI 0.76 to 1.14)
  firm_prior                 n= 4031  events= 524  HR 1.88 (95% CI 1.47 to 2.44)

upper_bound (outcome: any; failed refits 0)
  design                     n=  969  events= 125  HR 1.31 (95% CI 0.99 to 1.61)
  nondesign                  n=  999  events=  88  HR 1.04 (95% CI 0.72 to 1.38)
  distant                    n= 2914  events= 183  HR 0.92 (95% CI 0.74 to 1.12)
  firm_prior     

## 6. Kaplan-Meier cumulative incidence of own recall, by exposure group (FDA labeling)

In [7]:
def km(t, e, at):
    t, e = np.asarray(t), np.asarray(e); s, out = 1.0, {a: 1.0 for a in at}
    for u in np.unique(t[e == 1]):
        s *= 1 - ((t == u) & (e == 1)).sum() / (t >= u).sum()
        for a in at:
            if u <= a: out[a] = s
    return {a: round(100 * (1 - out[a]), 1) for a in at}

KM = []
for grp in ["design", "nondesign", "distant", "reference"]:
    s = D[D.grp_fda == grp]; c = km(s.t_any, s.ev_any, [5, 10])
    KM.append({"group": grp, "n": len(s), "events": int(s.ev_any.sum()),
               "cuminc_5y": c[5], "at_risk_5y": int((s.t_any >= 5).sum()),
               "cuminc_10y": c[10], "at_risk_10y": int((s.t_any >= 10).sum())})
KM = pd.DataFrame(KM); KM.to_csv("data/design_harm_km.csv", index=False); print(KM.to_string(index=False))

    group    n  events  cuminc_5y  at_risk_5y  cuminc_10y  at_risk_10y
   design  438      60       14.0         251        16.4          100
nondesign 1530     153        7.8        1057        11.1          532
  distant 2914     183        5.5        1858         7.3          848
reference 4046     456        6.0        3484         9.7         2837


## 7. Registered interpretation rules and summary

In [8]:
p = RESULTS["primary"]["terms"]["design"]; u = RESULTS["upper_bound"]["terms"]["design"]
ppv = VALIDATION["ppv_fda_design"][2]
RULES = {
    "label_reliable (PPV >= 0.80)": bool(ppv >= 0.80),
    "primary_hr_above_1": bool(p["hr"] > 1), "primary_lower_ci_above_1": bool(p["ci"][0] > 1),
    "upper_bound_hr_above_1": bool(u["hr"] > 1),
}
RULES["primary_hypothesis"] = ("inconclusive (FDA label unreliable)" if not RULES["label_reliable (PPV >= 0.80)"] else
                               "supported" if (RULES["primary_hr_above_1"] and RULES["primary_lower_ci_above_1"]
                                               and RULES["upper_bound_hr_above_1"]) else "not supported")
SUMMARY = {
    "plan": "Design-Recall Harm Analysis Plan (Option A) v1.0, 2026-11-12", "seed": SEED, "n_boot": NBOOT,
    "censor_date": str(SNAP.date()), "population": int(len(D)),
    "excluded_missing_dates": n_bad_dates, "excluded_nonpositive_followup": n_nonpos,
    "families": int(len(fams)),
    "groups_fda": D.grp_fda.value_counts().to_dict(), "groups_upper": D.grp_upper.value_counts().to_dict(),
    "groups_hand": D.grp_hand.value_counts().to_dict(),
    "events_any": int(D.ev_any.sum()), "events_ci_ii": int(D.ev_ci_ii.sum()), "events_design": int(D.ev_design.sum()),
    "validation": VALIDATION, "models": RESULTS, "km": KM.to_dict(orient="records"), "rules": RULES,
}
json.dump(SUMMARY, open("data/design_harm_summary.json", "w"), indent=2)
print(json.dumps(RULES, indent=2))
print("CHECKPOINT  wrote data/design_harm_summary.json, data/design_harm_devices.csv, data/design_harm_km.csv")

{
  "label_reliable (PPV >= 0.80)": true,
  "primary_hr_above_1": true,
  "primary_lower_ci_above_1": true,
  "upper_bound_hr_above_1": true,
  "primary_hypothesis": "supported"
}
CHECKPOINT  wrote data/design_harm_summary.json, data/design_harm_devices.csv, data/design_harm_km.csv
